# TDR Explorer

An interactive app for working with **Terra Data Repository snapshots** — or, for data custodians, **TDR datasets** — from a notebook. Run all cells (`Cell ▸ Run All` or `Run ▸ Run All Cells`) and use the app that appears at the bottom.

The app launches in **snapshot mode** by default; data custodians can start it in **dataset mode** instead with `TdrExplorer(mode="datasets").show()`. The descriptions below are written for snapshot mode — in dataset mode everything works the same way with datasets in place of snapshots, except: datasets have no DUOS links, dataset schemas don't report row counts, and query results include TDR metadata columns like `datarepo_row_id`.

**What it does**

| Tab | Purpose |
|---|---|
| **Snapshots** | Search the TDR snapshots your Terra identity can read. Nothing loads until you click **Search**; a blank keyword filter returns *all* your snapshots (can be slow if you have thousands). Comma-separate keywords to search for any of them at once (e.g. `emerge, 1000G` runs one TDR search per keyword and combines the unique results). Results are numbered and paginated (100 per page). Snapshots registered with DUOS show a clickable link to their DUOS dataset page for consent/usage details. Search results also populate the snapshot pickers on the other tabs. |
| **Schema** | Browse tables, columns, and row counts for a snapshot (type in the picker to filter), with a link to open its BigQuery dataset directly in BigQuery Studio. |
| **Query** | Run SQL in BigQuery and export the results, in one of two modes. **Single query**: use *Insert table ref* to drop a fully qualified table name wherever you put `??` in your SQL. **Fan-out across snapshots**: write the SQL once with `{dataset}` as the dataset, select multiple snapshots, and the app runs it per snapshot and combines results into one DataFrame (`snapshot_name` and `snapshot_id` columns are added; differing schemas are unioned, with missing columns as NaN). Each mode keeps its own SQL when you toggle between them. Failures are isolated in both Dry run and Run: if a snapshot errors (e.g. it lacks the queried table), the rest still proceed and you get the combined result plus a note of what failed. After a successful run, the **Export result…** button opens a panel to save the downloaded result as a file (`.csv`/`.tsv`/`.parquet`/`.json`, inferred from the path extension) in the workspace bucket or any `gs://` URL you can write to, or to upsert it into a **Terra data table** in this or any workspace where you have writer access. |
| **Log** | A running record of every query, dry run, and export (time, status, rows, GB scanned, SQL/destination). Persisted to the environment's disk so it survives pause/resume; can be saved to the workspace bucket or cleared. |

**Requirements**: a GCP-backed Terra cloud environment (the default Jupyter environment works as-is — no installs needed).

**Tips**
- Use **Dry run** before big queries: it validates the SQL and reports estimated bytes scanned plus an approximate on-demand cost ($6.25/TiB; the first 1 TiB per month per billing project is free). Queries bill to this workspace's Google project. In **dataset mode**, dry-run estimates show 0 GB — BigQuery can't pre-estimate scans of external-table-backed data and reports a lower bound, *not* a free query; the actual scan size and cost are shown after each real run and recorded in the Log.
- Every run's output starts with a header showing the mode and the SQL that was executed, so results are always self-describing.
- The Query tab previews the first 100 rows of each result by default; adjust with `TdrExplorer(preview_rows=...)` (up to 1,000 — beyond that, notebook tables get slow, so use Export or pandas instead). The full result is always available as `app.result_df`.
- For data-table export, pick a result column with unique values as the **ID column**, or let the app auto-number rows (zero-padded so they sort correctly in Terra). Export paths are validated for GCS-safety (no spaces, wildcards, or '#').
- **Data custodians**: launch with `TdrExplorer(mode="datasets").show()` to browse and query your **TDR datasets** instead of snapshots.
- **Big results**: DataFrame downloads are capped at 1,000,000 rows per query to protect notebook memory (queries themselves are never truncated — only the download). For larger results, use **Run → bucket**, which has BigQuery write the full result set directly to the workspace bucket (CSV, JSON, or Parquet) without touching notebook memory. Results with ARRAY columns can't be represented as CSV; the app automatically falls back to Parquet for those. Power users can raise the cap (`TdrExplorer(max_download_rows=...)`) or add a per-query cost ceiling (`TdrExplorer(max_gb_billed=...)`) at launch. Bucket exports can be merged into a single file up to 50 GB (`TdrExplorer(max_merge_gb=...)`); larger ones stay sharded, which readers accept via wildcard anyway. CSV bucket exports flatten ARRAY/STRUCT columns to text by default (arrays joined by a separator, structs as JSON) so you get real CSV; untick that option to export such results as Parquet with the nested types intact.
- Data-table exports are blocked above 50,000 rows — Terra data tables aren't built for that scale; use a bucket CSV instead.


## App code

You don't need to read or modify anything below — just run it. (In Jupyter you can collapse this cell with the blue bar to its left.)


In [ ]:
"""
TDR Snapshot Explorer
=====================
An ipywidgets app for Terra notebooks that lets users:

  1. List/search the Terra Data Repository (TDR) snapshots — or, with
     mode="datasets", TDR datasets — they can access
  2. Browse schemas (tables + columns, via the TDR API)
  3. Run SQL against their BigQuery datasets — either a single query,
     or the same query fanned out across many snapshots/datasets and
     combined into
     one DataFrame (tolerant of schema differences)
  4. Export results to a file (CSV/TSV/Parquet/JSON, inferred from the
     path extension) in the workspace bucket, or to a Terra
     data table

Assumes a GCP-backed Terra cloud environment.
"""

import os
import re
import json
import gzip
import subprocess
import html
import datetime
import requests
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed

import google.auth
from google.auth.transport.requests import Request as _GAuthRequest
from google.cloud import bigquery

import ipywidgets as w
from IPython.display import display, HTML

TDR_BASE = os.environ.get("TDR_BASE_URL", "https://data.terra.bio")

# Terra notebook environment variables
WORKSPACE_NAMESPACE = os.environ.get("WORKSPACE_NAMESPACE", "")
WORKSPACE_NAME = os.environ.get("WORKSPACE_NAME", "")
WORKSPACE_BUCKET = os.environ.get("WORKSPACE_BUCKET", "")  # e.g. gs://fc-...
GOOGLE_PROJECT = os.environ.get("GOOGLE_PROJECT", "")      # billing project for BQ jobs

# BigQuery on-demand pricing (US regions), for dry-run estimates only.
# First 1 TiB processed per billing project per month is free.
BQ_PRICE_PER_TIB = 6.25

# Marker users can place in SQL to control where "Insert table ref" lands.
INSERT_MARKER = "??"

FANOUT_PLACEHOLDER = "{dataset}"  # replaced with project.dataset per snapshot

# Activity log persists here (JSONL). The home directory sits on Terra's
# persistent disk, so the log survives cloud environment pause/resume.
LOG_PATH = os.path.expanduser("~/tdr_explorer_log.jsonl")


# ---------------------------------------------------------------------------
# Auth + TDR API helpers
# ---------------------------------------------------------------------------

_SCOPES = [
    "https://www.googleapis.com/auth/cloud-platform",
    "https://www.googleapis.com/auth/userinfo.email",
    "https://www.googleapis.com/auth/userinfo.profile",
]


def _access_token() -> str:
    """Get a fresh access token from the notebook's application default creds."""
    creds, _ = google.auth.default(scopes=_SCOPES)
    creds.refresh(_GAuthRequest())
    return creds.token


def tdr_get(path: str, **params):
    """GET against the TDR API, raising on error."""
    resp = requests.get(
        f"{TDR_BASE}{path}",
        headers={"Authorization": f"Bearer {_access_token()}"},
        params=params,
    )
    resp.raise_for_status()
    return resp.json()


def list_resources(resource: str, filter_text: str = None,
                   page_size: int = 200) -> list:
    """Enumerate TDR snapshots or datasets the caller can read, with
    optional server-side text filtering (matches name/description).
    resource: 'snapshots' or 'datasets'."""
    items, offset = [], 0
    while True:
        params = {"offset": offset, "limit": page_size,
                  "sort": "name", "direction": "asc"}
        if filter_text:
            params["filter"] = filter_text
        page = tdr_get(f"/api/repository/v1/{resource}", **params)
        batch = page.get("items", [])
        items.extend(batch)
        if len(batch) < page_size:
            return items
        offset += page_size


def get_resource_detail(resource: str, resource_id: str) -> dict:
    """Fetch one snapshot/dataset with table schemas and BigQuery access
    info. Snapshots expose tables via include=TABLES; datasets nest them
    under schema via include=SCHEMA."""
    include = (["TABLES", "ACCESS_INFORMATION"] if resource == "snapshots"
               else ["SCHEMA", "ACCESS_INFORMATION"])
    return tdr_get(f"/api/repository/v1/{resource}/{resource_id}",
                   include=include)


def _estimate_cost(total_bytes: int) -> str:
    tib = total_bytes / 2**40
    return (f"~{total_bytes / 1e9:,.3f} GB scanned "
            f"(≈ ${tib * BQ_PRICE_PER_TIB:,.4f} at "
            f"${BQ_PRICE_PER_TIB}/TiB on-demand; first 1 TiB per "
            f"billing project per month is free)")


class _MergeTooLarge(Exception):
    def __init__(self, nbytes, cap_gb):
        super().__init__(
            f"merged file would be ~{nbytes / 1e9:,.1f} GB, over the "
            f"{cap_gb:,.0f} GB merge limit")
        self.nbytes, self.cap_gb = nbytes, cap_gb


def _is_array_csv_error(e) -> bool:
    """True if this error is 'CSV can't hold ARRAY/STRUCT columns'."""
    msg = str(e).lower()
    return ("only simple types" in msg
            or ("csv" in msg and ("nested" in msg or "repeated" in msg)))


_GCS_FORBIDDEN_CHARS = set(" \t\n\r*?[]#")


def validate_gcs_path(path: str) -> str:
    """Return an error message if the path isn't safe to use as a GCS
    destination, or None if it's fine. Deliberately stricter than what
    GCS technically allows (e.g. spaces are legal but break gsutil
    quoting and downstream tools; wildcards would corrupt EXPORT DATA
    URIs and gsutil globbing)."""
    if not path:
        return "Path is empty."
    for ch in path:
        if ch in _GCS_FORBIDDEN_CHARS or ord(ch) < 32:
            shown = repr(ch) if ord(ch) < 32 else f"'{ch}'"
            return (f"Path contains {shown}, which isn't allowed here — "
                    "no spaces, wildcards (* ? [ ]), or '#'.")
    body = path
    if path.startswith("gs://"):
        bucket, _, body = path[5:].partition("/")
        if not re.fullmatch(r"[a-z0-9][a-z0-9._-]{1,61}[a-z0-9]", bucket):
            return (f"'{bucket}' is not a valid GCS bucket name (lowercase "
                    "letters, digits, dashes, dots, underscores; 3-63 "
                    "chars).")
    if "//" in body:
        return "Path contains an empty segment ('//')."
    return None


# ---------------------------------------------------------------------------
# The app
# ---------------------------------------------------------------------------

class TdrExplorer:
    def __init__(self, billing_project: str = None,
                 max_download_rows: int = 1_000_000,
                 max_gb_billed: float = None,
                 mode: str = "snapshots",
                 preview_rows: int = 100,
                 max_merge_gb: float = 50):
        """
        max_download_rows: cap on rows pulled into the notebook as a
            DataFrame. Queries still run fully in BigQuery; only the
            download is truncated (with a warning). Raise or lower per
            your VM's memory: TdrExplorer(max_download_rows=5_000_000).
        max_gb_billed: optional hard cap on bytes scanned per query.
            Queries estimated to exceed it are rejected by BigQuery
            BEFORE running (no charge). E.g. max_gb_billed=100.
        mode: 'snapshots' (default) or 'datasets'. Dataset mode is for
            data custodians querying TDR datasets directly; note that
            dataset tables include TDR metadata columns (datarepo_row_id)
            and dataset schemas don't report row counts.
        preview_rows: how many result rows to render in the Query tab
            preview (clamped to 10-1,000). The preview is display-only;
            the full result is always in .result_df regardless. Beyond
            ~1,000 rows, notebook HTML tables get slow and can exceed
            Jupyter's output message limits — use Export or .result_df
            to work with more.
        max_merge_gb: largest bucket export the app will merge into a
            single file. Bigger exports are left as shards, which stay
            readable by wildcard and can be read in parallel. Cloud
            Storage's own hard limit is 5 TiB per object, but merging
            at that scale produces a file nothing can read efficiently.
        """
        if mode not in ("snapshots", "datasets"):
            raise ValueError("mode must be 'snapshots' or 'datasets'")
        self.mode = mode
        self.noun = mode[:-1]          # 'snapshot' / 'dataset'
        self.Noun = self.noun.capitalize()
        self.preview_rows = max(10, min(int(preview_rows), 1000))
        # Cloud Storage caps any object at 5 TiB; stay well under it.
        self.max_merge_gb = max(0.1, min(float(max_merge_gb), 5000))
        self.billing_project = billing_project or GOOGLE_PROJECT
        self.max_download_rows = max_download_rows
        self.max_gb_billed = max_gb_billed
        self.bq = bigquery.Client(project=self.billing_project)
        self.snapshots = []          # summaries from the last listing
        self.label_to_id = {}        # display label -> resource id
        self.label_to_name = {}      # display label -> resource name
        self.detail_cache = {}       # snapshot_id -> full detail
        self.result_df = None        # last query result
        self._last_capped = False    # was the last download truncated?
        self._search_note = []       # per-keyword search summary
        self._snap_page = 0          # snapshots table pagination
        self.log = self._load_log()  # activity log (queries + exports)

        self._build_ui()

    # -- activity log ---------------------------------------------------------

    @staticmethod
    def _load_log() -> list:
        entries = []
        try:
            with open(LOG_PATH) as f:
                for line in f:
                    line = line.strip()
                    if line:
                        entries.append(json.loads(line))
        except FileNotFoundError:
            pass
        except Exception:
            pass  # a corrupt log should never break the app
        return entries

    def _log(self, kind: str, detail: str, status: str,
             rows=None, gb=None):
        entry = {
            "time": datetime.datetime.now().isoformat(timespec="seconds"),
            "mode": self.mode,
            "kind": kind,
            "status": status,
            "rows": rows,
            "gb": round(gb, 3) if gb is not None else None,
            "detail": detail[:1000],
        }
        self.log.append(entry)
        try:
            with open(LOG_PATH, "a") as f:
                f.write(json.dumps(entry) + "\n")
        except Exception:
            pass  # logging must never break the operation being logged

    @property
    def log_df(self) -> pd.DataFrame:
        cols = ["time", "mode", "kind", "status", "rows", "gb", "detail"]
        if not self.log:
            return pd.DataFrame(columns=cols)
        # reindex tolerates entries persisted before the mode column existed
        df = pd.DataFrame(self.log).reindex(columns=cols)
        df["mode"] = df["mode"].fillna("")
        return df.iloc[::-1].reset_index(drop=True)

    def _log_with_failures(self, kind: str, detail: str, ok_count: int,
                           failures: list, rows=None, gb=None):
        """Log an operation, embedding per-item failure messages so the
        log is useful for diagnosis, not just counting."""
        if not failures:
            status = "ok"
        elif ok_count:
            status = (f"partial — {len(failures)} failed "
                      f"(first: {failures[0][0]}: {failures[0][1][:120]})")
        else:
            status = (f"failed "
                      f"({failures[0][0]}: {failures[0][1][:150]})")
        if failures:
            detail += "\n-- FAILURES --\n" + "\n".join(
                f"{lb}: {msg[:200]}" for lb, msg in failures)
        self._log(kind, detail, status, rows=rows, gb=gb)

    # -- shared helpers -----------------------------------------------------

    def _detail(self, resource_id: str) -> dict:
        if resource_id not in self.detail_cache:
            self.detail_cache[resource_id] = \
                get_resource_detail(self.mode, resource_id)
        return self.detail_cache[resource_id]

    def _gsutil(self, args: str, capture=False):
        """Run gsutil, billing requests to the workspace project so
        requester-pays buckets work."""
        u = f"-u '{GOOGLE_PROJECT}' " if GOOGLE_PROJECT else ""
        cmd = f"gsutil {u}{args}"
        if not capture:
            return os.system(cmd), ""
        p = subprocess.run(cmd, shell=True, capture_output=True, text=True)
        return p.returncode, (p.stdout if p.returncode == 0 else p.stderr)

    def _result_schema(self, sql: str) -> list:
        """Result schema fields for a query, via a free dry run."""
        job = self.bq.query(sql, job_config=self._job_config(
            dry_run=True, use_query_cache=False))
        return list(job.schema or [])

    def _result_columns(self, sql: str) -> list:
        """Column names of a query's result, via a free dry run."""
        return [f.name for f in self._result_schema(sql)]

    def _csv_unsafe_fields(self, sql: str) -> list:
        """Fields CSV can't represent: ARRAY (REPEATED) or STRUCT
        (RECORD). Returns [(name, field_type, mode), ...], or [] if the
        schema can't be determined."""
        try:
            fields = self._result_schema(sql)
        except Exception:
            return []
        bad = []
        for f in fields:
            mode = (getattr(f, "mode", "") or "").upper()
            ftype = (getattr(f, "field_type", "") or "").upper()
            if mode == "REPEATED" or ftype in ("RECORD", "STRUCT"):
                bad.append((f.name, ftype, mode))
        return bad

    @staticmethod
    def _flatten_sql(sql: str, bad_fields: list, delim: str) -> str:
        """Rewrite a query so CSV-incompatible columns become strings,
        keeping every other column and its position via SELECT * REPLACE.

        Scalar arrays become delimiter-joined text ('a|b'); anything
        involving a STRUCT becomes JSON, which is the only faithful
        single-cell representation."""
        lit = delim.replace("'", "''")
        repls = []
        for name, ftype, mode in bad_fields:
            col = f"`{name}`"
            if ftype in ("RECORD", "STRUCT"):
                expr = f"TO_JSON_STRING({col})"
            else:
                expr = (f"ARRAY_TO_STRING(ARRAY(SELECT CAST(e AS STRING) "
                        f"FROM UNNEST({col}) AS e), '{lit}')")
            repls.append(f"{expr} AS {col}")
        inner = sql.strip().rstrip(";")
        return (f"SELECT * REPLACE({', '.join(repls)}) FROM (\n"
                f"{inner}\n)")

    def _combine_shards(self, base, safe, ext, fmt, sql):
        """Merge an export's shards into ONE object using GCS compose,
        which happens server-side (no download, no egress). Works for
        CSV and newline-delimited JSON because those concatenate
        cleanly — and concatenated gzip members are themselves a valid
        gzip stream (RFC 1952 defines a gzip file as a series of members,
        so gzip/zcat/pandas/R all read the merged file transparently).
        Returns (final_uri, warning_or_None).

        Notes on GCS compose: at most 32 sources per call (so this folds
        in batches), sources must share a bucket and storage class, and
        composite objects carry a CRC32C but no MD5 — tools that expect
        an MD5 in the ETag may warn. Source shards are deleted after a
        successful merge so they don't linger on the bill."""
        rc, out = self._gsutil(f"ls -l '{base}/{safe}_*.{ext}'",
                               capture=True)
        if rc != 0:
            raise RuntimeError(f"couldn't list shards: {out.strip()[:200]}")
        shards, total = [], 0
        for line in out.splitlines():
            fields = line.split()
            if len(fields) < 2 or not fields[0].isdigit():
                continue          # skips the trailing TOTAL: line
            total += int(fields[0])
            shards.append(fields[-1])
        shards.sort()
        if not shards:
            raise RuntimeError("no shards found to merge")
        if total > self.max_merge_gb * 1e9:
            raise _MergeTooLarge(total, self.max_merge_gb)

        parts, cleanup = list(shards), list(shards)
        # CSV shards are written headerless so they concatenate; put a
        # one-line header object in front of them.
        if fmt == "CSV":
            cols = self._result_columns(sql)
            if not cols:
                raise RuntimeError("couldn't determine column names")
            local = "_tdr_export_header.csv.gz"
            with gzip.open(local, "wt") as fh:
                fh.write(",".join(cols) + "\n")
            header_uri = f"{base}/{safe}__header.csv.gz"
            rc, out = self._gsutil(f"cp '{local}' '{header_uri}'")
            if rc != 0:
                raise RuntimeError("couldn't upload header object")
            total += os.path.getsize(local)
            parts = [header_uri] + parts
            cleanup.append(header_uri)

        final = f"{base}/{safe}.{ext}"
        # compose takes at most 32 sources per call, so fold in batches
        batch_i = 0
        while len(parts) > 1:
            if len(parts) <= 32:
                srcs, parts = parts, []
                dest = final
            else:
                srcs, parts = parts[:32], parts[32:]
                dest = f"{base}/{safe}__part{batch_i}.{ext}"
                batch_i += 1
                cleanup.append(dest)
            quoted = " ".join(f"'{p}'" for p in srcs)
            # compose doesn't infer a content type, so set it explicitly;
            # otherwise the merged object can end up mislabelled.
            ctype = ("application/gzip" if ext.endswith(".gz")
                     else "application/octet-stream")
            rc, out = self._gsutil(
                f"-h 'Content-Type:{ctype}' compose {quoted} '{dest}'",
                capture=True)
            if rc != 0:
                raise RuntimeError(f"compose failed: {out.strip()[:200]}")
            parts = [dest] + parts
        if parts and parts[0] != final:
            self._gsutil(f"mv '{parts[0]}' '{final}'")
            cleanup = [c for c in cleanup if c != parts[0]]

        # Compose is a byte-for-byte concatenation, so the merged object
        # must weigh exactly what its parts did. A mismatch would mean
        # something transformed the bytes (e.g. transcoding), which for
        # gzip would mean a corrupt file — so check before deleting.
        warn = None
        rc, out = self._gsutil(f"ls -l '{final}'", capture=True)
        merged_bytes = 0
        for line in out.splitlines():
            fields = line.split()
            if len(fields) >= 2 and fields[0].isdigit():
                merged_bytes = int(fields[0])
                break
        if rc != 0 or merged_bytes != total:
            warn = (f"merged file is {merged_bytes:,} bytes but its parts "
                    f"totalled {total:,} — shards were NOT deleted so you "
                    "can verify")
            return final, warn

        stale = [c for c in cleanup if c != final]
        if stale:
            self._gsutil("-m rm " + " ".join(f"'{c}'" for c in stale))
        return final, None

    def _tables_of(self, detail: dict) -> list:
        """Table schema list, wherever this resource type keeps it."""
        if self.mode == "datasets":
            return (detail.get("schema") or {}).get("tables", [])
        return detail.get("tables", [])

    def _job_config(self, **kwargs) -> bigquery.QueryJobConfig:
        if self.max_gb_billed is not None:
            kwargs["maximum_bytes_billed"] = int(self.max_gb_billed * 1e9)
        return bigquery.QueryJobConfig(**kwargs)

    def _bq_info(self, snapshot_id: str) -> dict:
        """Return {'projectId', 'datasetName', 'tables': {name: qualifiedName}}."""
        detail = self._detail(snapshot_id)
        bq = (detail.get("accessInformation") or {}).get("bigQuery")
        if not bq:
            raise RuntimeError(
                f"{self.Noun} {detail.get('name')} has no BigQuery access "
                "info — you may not have access to its data"
            )
        return {
            "projectId": bq["projectId"],
            "datasetName": bq["datasetName"],
            "tables": {t["name"]: t.get("qualifiedName",
                       f"{bq['projectId']}.{bq['datasetName']}.{t['name']}")
                       for t in bq.get("tables", [])},
        }

    def _rebuild_labels(self):
        """Map display labels -> ids, disambiguating duplicate names."""
        self.label_to_id = {}
        self.label_to_name = {}
        name_counts = {}
        for s in self.snapshots:
            name_counts[s["name"]] = name_counts.get(s["name"], 0) + 1
        for s in self.snapshots:
            label = s["name"]
            if name_counts[s["name"]] > 1:
                label = f"{s['name']} [{s['id'][:8]}]"
            self.label_to_id[label] = s["id"]
            self.label_to_name[label] = s["name"]

    def _sync_selectors(self):
        labels = sorted(self.label_to_id.keys())
        placeholder = "type to filter…" if labels else \
            f"run a Search on the {self.Noun}s tab first…"
        for cb in (self.schema_snap_cb, self.query_snap_cb):
            cb.options = labels
            cb.placeholder = placeholder
        self._filter_fanout_options()

    def _refresh_snapshots(self, _=None):
        with self.snap_out:
            self.snap_out.clear_output()
            raw = self.snap_search.value.strip()
            self._search_note = []
            # comma-separated keywords are OR'd: one TDR call each,
            # results unioned by id
            keywords, seen = [], set()
            for k in (p.strip() for p in raw.split(",")):
                if k and k.lower() not in seen:
                    seen.add(k.lower())
                    keywords.append(k)

            if len(keywords) <= 1:
                filt = keywords[0] if keywords else None
                print(f"Loading {self.mode} from TDR"
                      + (f" matching '{filt}'" if filt else "") + " ...")
                try:
                    self.snapshots = list_resources(self.mode,
                                                    filter_text=filt)
                except Exception as e:
                    self.snap_out.clear_output()
                    print(f"Failed to list {self.mode}: {e}")
                    return
            else:
                print(f"Loading {self.mode} from TDR matching "
                      + " or ".join(f"'{k}'" for k in keywords) + " ...")
                by_id, kw_failures = {}, []
                self._search_note = []
                with ThreadPoolExecutor(max_workers=4) as pool:
                    futures = [(k, pool.submit(list_resources, self.mode,
                                               filter_text=k))
                               for k in keywords]
                    for k, fut in futures:
                        try:
                            items = fut.result()
                            note = f"  '{k}': {len(items)} match(es)"
                            for s in items:
                                by_id[s["id"]] = s
                        except Exception as e:
                            kw_failures.append(k)
                            note = f"  '{k}': FAILED — {e}"
                        print(note)
                        self._search_note.append(note)
                if kw_failures and not by_id:
                    print("All keyword searches failed.")
                    return
                self.snapshots = sorted(
                    by_id.values(),
                    key=lambda s: (s.get("name") or "").lower())
                print(f"  → {len(self.snapshots)} unique after combining")
            self._rebuild_labels()
        self._snap_page = 0
        self._render_snapshot_page()
        self._sync_selectors()

    SNAP_PAGE_SIZE = 100

    def _render_snapshot_page(self):
        """Render the current page of the search results table and keep
        the pager controls in sync."""
        n = len(self.snapshots)
        pages = max(1, -(-n // self.SNAP_PAGE_SIZE))  # ceil div
        self._snap_page = max(0, min(self._snap_page, pages - 1))
        start = self._snap_page * self.SNAP_PAGE_SIZE
        end = min(start + self.SNAP_PAGE_SIZE, n)

        with self.snap_out:
            self.snap_out.clear_output()
            for note in self._search_note:
                print(note)
            if not self.snapshots:
                print(f"No {self.mode} matched.")
            else:
                print(f"{n} {self.noun}(s)"
                      + (" (unique across keywords)"
                         if len(self._search_note) > 1 else "")
                      + f" — showing {start + 1}–{end}")

                def duos_link(s):
                    duos = s.get("duosId") or \
                        (s.get("duosFirecloudGroup") or {}).get("duosId")
                    if not duos:
                        return ""
                    return (f'<a href="https://duos.org/dataset/'
                            f'{html.escape(str(duos))}" target="_blank">'
                            f'{html.escape(str(duos))}</a>')

                def esc(v):
                    return html.escape(str(v)) if v else ""

                rows = []
                for i in range(start, end):
                    s = self.snapshots[i]
                    row = {
                        "#": i + 1,
                        "name": esc(s.get("name")),
                        "id": esc(s.get("id")),
                        "created": esc((s.get("createdDate") or "")[:19]),
                    }
                    if self.mode == "snapshots":
                        row["DUOS"] = duos_link(s)
                    row["description"] = esc((s.get("description")
                                              or "")[:80])
                    rows.append(row)
                display(HTML(pd.DataFrame(rows).to_html(escape=False,
                                                        index=False)))

        multi_page = pages > 1
        self.snap_pager.layout.display = "" if multi_page else "none"
        if multi_page:
            self.snap_prev.disabled = self._snap_page == 0
            self.snap_next.disabled = self._snap_page >= pages - 1
            self.snap_page_lbl.value = (
                f"&nbsp;page {self._snap_page + 1} of {pages}&nbsp;")

    def _snap_page_step(self, delta: int):
        self._snap_page += delta
        self._render_snapshot_page()

    # -- Tab 1: snapshots ----------------------------------------------------

    def _build_snapshots_tab(self):
        self.snap_search = w.Text(
            placeholder=f"keywords, comma = OR (blank = ALL {self.mode})",
            tooltip=f"Server-side match against {self.noun} name/description. "
                    "Comma-separate keywords to OR them, e.g. "
                    "'emerge, 1000G' (one TDR search per keyword, results "
                    f"combined). Leave blank to list ALL {self.mode} you "
                    "can access (may be slow if you have thousands).",
            layout=w.Layout(width="45%"))
        search_btn = w.Button(
            description="Search", icon="search", button_style="primary",
            tooltip=f"Query TDR for matching {self.mode}. A blank keyword "
                    f"filter returns ALL {self.mode} you can access.")
        search_btn.on_click(self._refresh_snapshots)
        note = w.HTML(
            "<i>Nothing loads until you click Search. The search results "
            f"also populate the {self.noun} pickers on the other tabs — "
            f"handy if you have thousands of {self.mode}. Comma-separate "
            "keywords to search for any of them at once.</i>")
        self.snap_out = w.Output(
            layout=w.Layout(max_height="500px", overflow="auto",
                            border="1px solid #ddd"))
        self.snap_prev = w.Button(description="◀ Prev",
                                  layout=w.Layout(width="90px"))
        self.snap_next = w.Button(description="Next ▶",
                                  layout=w.Layout(width="90px"))
        self.snap_page_lbl = w.HTML("")
        self.snap_prev.on_click(lambda _: self._snap_page_step(-1))
        self.snap_next.on_click(lambda _: self._snap_page_step(+1))
        self.snap_pager = w.HBox([self.snap_prev, self.snap_page_lbl,
                                  self.snap_next])
        self.snap_pager.layout.display = "none"
        return w.VBox([w.HBox([self.snap_search, search_btn]), note,
                       self.snap_pager, self.snap_out])

    # -- Tab 2: schema explorer ----------------------------------------------

    def _build_schema_tab(self):
        self.schema_snap_cb = w.Combobox(
            description=f"{self.Noun}:",
            placeholder=f"run a Search on the {self.Noun}s tab first…",
            ensure_option=True, layout=w.Layout(width="60%"))
        self.schema_out = w.Output(
            layout=w.Layout(max_height="500px", overflow="auto",
                            border="1px solid #ddd"))
        self.schema_snap_cb.observe(self._show_schema, names="value")
        return w.VBox([self.schema_snap_cb, self.schema_out])

    def _show_schema(self, change):
        resource_id = self.label_to_id.get(change["new"])
        if not resource_id:
            return
        with self.schema_out:
            self.schema_out.clear_output()
            try:
                detail = self._detail(resource_id)
            except Exception as e:
                print(f"Failed to load {self.noun}: {e}")
                return
            try:
                info = self._bq_info(resource_id)
                proj, dsn = info["projectId"], info["datasetName"]
                console = ("https://console.cloud.google.com/bigquery"
                           f"?p={proj}&d={dsn}&page=dataset"
                           + (f"&project={self.billing_project}"
                              if self.billing_project else ""))
                display(HTML(
                    f"<b>BigQuery dataset:</b> <code>{proj}.{dsn}</code> "
                    f'&nbsp;<a href="{console}" target="_blank">open in '
                    "BigQuery Studio ↗</a>"))
                if self.mode == "datasets":
                    display(HTML(
                        "<i>Note: TDR dataset tables also carry metadata "
                        "columns (e.g. <code>datarepo_row_id</code>) not "
                        "shown in the schema below, and row counts aren't "
                        "reported by the datasets API.</i>"))
            except RuntimeError as e:
                print(e)
            for table in sorted(self._tables_of(detail),
                                key=lambda t: (t.get("name") or "").lower()):
                rows = [{"column": c["name"],
                         "type": c["datatype"],
                         "array": c.get("array_of", False)}
                        for c in table.get("columns", [])]
                rc = table.get("rowCount")
                count_txt = f" ({rc:,} rows)" if rc is not None else ""
                display(HTML(f"<h4>{table['name']}{count_txt}</h4>"))
                display(pd.DataFrame(rows))

    # -- Tab 3: query --------------------------------------------------------

    def _build_query_tab(self):
        self.query_mode = w.ToggleButtons(
            options=["Single query", f"Fan-out across {self.mode}"],
            tooltips=["One SQL statement, fully qualified table names",
                      f"Run the same SQL once per selected {self.noun} and "
                      "concatenate the results into one DataFrame"])

        # single-query controls
        self.query_snap_cb = w.Combobox(
            description=f"{self.Noun}:",
            placeholder=f"run a Search on the {self.Noun}s tab first…",
            ensure_option=True, layout=w.Layout(width="42%"))
        self.query_table_dd = w.Dropdown(description="Table:",
                                         layout=w.Layout(width="35%"))
        insert_btn = w.Button(description="Insert table ref", icon="plus",
                              tooltip=f"Replaces the first '{INSERT_MARKER}' "
                                      "in your SQL, or appends if absent")
        single_help = w.HTML(
            f"<i>Pick a {self.noun} and table, then <b>Insert table ref</b> "
            f"drops the fully qualified table name at the first "
            f"<code>{INSERT_MARKER}</code> in your SQL (or appends to the "
            "end if there isn't one).</i>")
        self.single_box = w.VBox([single_help,
                                  w.HBox([self.query_snap_cb,
                                          self.query_table_dd, insert_btn])])

        # fan-out controls
        self.fanout_filter = w.Text(
            placeholder=f"filter {self.noun} list…",
            layout=w.Layout(width="40%"))
        self.fanout_select = w.SelectMultiple(
            description=f"{self.Noun}s:", rows=8,
            layout=w.Layout(width="60%"))
        fanout_help = w.HTML(
            f"<i>Write your SQL once using <code>{FANOUT_PLACEHOLDER}</code> "
            f"as the dataset, e.g. <code>SELECT * FROM "
            f"`{FANOUT_PLACEHOLDER}.person`</code>. It runs once per selected "
            f"{self.noun}; results are concatenated with <code>"
            f"{self.noun}_name</code> and <code>{self.noun}_id</code> "
            "columns added, and differing schemas are unioned (missing "
            "columns become NaN).</i>")
        self.fanout_box = w.VBox([fanout_help,
                                  w.HBox([self.fanout_filter]),
                                  self.fanout_select])
        self.fanout_box.layout.display = "none"
        self.fanout_filter.observe(
            lambda ch: self._filter_fanout_options(), names="value")

        # Each mode keeps its own SQL text, saved/restored when toggling.
        self._sql_state = {
            "single": f"SELECT * FROM {INSERT_MARKER} LIMIT 100",
            "fanout": (f"SELECT * FROM `{FANOUT_PLACEHOLDER}.table_name` "
                       "LIMIT 100"),
        }
        self.sql_box = w.Textarea(
            value=self._sql_state["single"],
            layout=w.Layout(width="98%", height="180px"))
        run_btn = w.Button(description="Run", icon="play",
                           button_style="success")
        self.export_btn = w.Button(
            description="Export result…", icon="save", disabled=True,
            tooltip="Export the downloaded result of the last run to a "
                    "file in a bucket or a Terra data table. Enabled "
                    "after a successful Run.")
        self.export_btn.on_click(self._toggle_export_panel)
        dry_btn = w.Button(description="Dry run (cost estimate)",
                           icon="dollar")
        bucket_btn = w.Button(
            description="Run → bucket…", icon="cloud-upload",
            tooltip="For huge results: BigQuery writes the FULL result "
                    "set directly to a bucket — nothing is downloaded to "
                    "the notebook, so there is no row limit and no memory "
                    "risk. Opens a panel to pick format and destination.")
        self.bucket_fmt_dd = w.Dropdown(
            options=["CSV (gzip)", "JSON (gzip)", "Parquet"],
            value="CSV (gzip)", description="Format:",
            layout=w.Layout(width="220px"),
            tooltip="CSV cannot represent ARRAY columns — the app "
                    "auto-falls back to Parquet for those, or pick "
                    "JSON/Parquet up front.")
        self.bucket_path = w.Text(
            description="Folder:", layout=w.Layout(width="60%"),
            tooltip="Destination folder for the export files. Relative "
                    "paths go under the workspace bucket; a full gs:// "
                    "URL writes to any bucket you can write to.")
        start_btn = w.Button(description="Start export", icon="play",
                             button_style="warning")
        cancel_btn = w.Button(description="Cancel")
        self.bucket_flatten_cb = w.Checkbox(
            value=True, indent=False,
            description="Flatten ARRAY/STRUCT columns to keep CSV",
            tooltip="CSV can't hold ARRAY or STRUCT columns. With this "
                    "on, they're converted to text (arrays joined by the "
                    "separator, structs as JSON) so you still get CSV. "
                    "With it off, such results are exported as Parquet "
                    "instead.")
        self.bucket_delim = w.Text(
            value="|", description="Sep:",
            layout=w.Layout(width="120px"),
            tooltip="Separator used when joining array values into one "
                    "CSV cell.")
        self.bucket_flatten_box = w.HBox([self.bucket_flatten_cb,
                                          self.bucket_delim])

        def _toggle_flatten_visibility(change=None):
            is_csv = self.bucket_fmt_dd.value.startswith("CSV")
            self.bucket_flatten_box.layout.display = "" if is_csv else "none"
        self.bucket_fmt_dd.observe(_toggle_flatten_visibility, names="value")

        self.bucket_single_cb = w.Checkbox(
            value=False, indent=False,
            description="Merge into a single file (CSV/JSON only)",
            tooltip="BigQuery always writes many shards. Ticking this "
                    "merges them into one file afterwards, inside Cloud "
                    "Storage — nothing is downloaded. Parquet can't be "
                    "merged this way and stays sharded, and very large "
                    "exports are left as shards too.")
        panel_help = w.HTML(
            "<i>BigQuery writes the full result set to this folder, "
            "sharded across many files. Tick the box below to merge them "
            "into a single file afterwards (CSV and JSON only; the merge "
            "happens inside Cloud Storage, so nothing is downloaded). "
            "Relative paths go under the workspace bucket; full "
            "<code>gs://</code> URLs work too.</i>")
        self.bucket_panel = w.VBox(
            [panel_help, w.HBox([self.bucket_fmt_dd]), self.bucket_path,
             self.bucket_flatten_box, self.bucket_single_cb,
             w.HBox([start_btn, cancel_btn])],
            layout=w.Layout(border="1px solid #f0ad4e", padding="8px",
                            margin="4px 0"))
        self.bucket_panel.layout.display = "none"

        def open_panel(_):
            showing = self.bucket_panel.layout.display != "none"
            if showing:
                self.bucket_panel.layout.display = "none"
                return
            stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
            self.bucket_path.value = f"exports/query_{stamp}"
            self.bucket_panel.layout.display = ""

        def close_panel(_=None):
            self.bucket_panel.layout.display = "none"

        bucket_btn.on_click(open_panel)
        cancel_btn.on_click(close_panel)
        start_btn.on_click(lambda _: (close_panel(),
                                      self._run_to_bucket(None)))
        self.query_out = w.Output(
            layout=w.Layout(max_height="500px", overflow="auto",
                            border="1px solid #ddd"))

        self.query_mode.observe(self._toggle_query_mode, names="value")
        self.query_snap_cb.observe(self._update_query_tables, names="value")
        insert_btn.on_click(self._insert_table_ref)
        run_btn.on_click(lambda _: self._run_query(dry=False))
        dry_btn.on_click(lambda _: self._run_query(dry=True))

        return w.VBox([self.query_mode, self.single_box, self.fanout_box,
                       self.sql_box,
                       w.HBox([run_btn, dry_btn, bucket_btn,
                               self.export_btn]),
                       self.bucket_panel,
                       self._build_export_panel(),
                       self.query_out])

    def _toggle_query_mode(self, change):
        old_key = "fanout" if change["old"].startswith("Fan-out") else "single"
        new_key = "fanout" if change["new"].startswith("Fan-out") else "single"
        # Save the SQL for the mode being left; restore the other mode's SQL.
        self._sql_state[old_key] = self.sql_box.value
        self.sql_box.value = self._sql_state[new_key]
        self.query_out.clear_output()  # avoid stale output from other mode
        fanout = new_key == "fanout"
        self.fanout_box.layout.display = "" if fanout else "none"
        self.single_box.layout.display = "none" if fanout else ""

    def _filter_fanout_options(self):
        filt = self.fanout_filter.value.strip().lower()
        labels = sorted(l for l in self.label_to_id
                        if filt in l.lower()) if filt else \
                 sorted(self.label_to_id)
        keep = tuple(v for v in self.fanout_select.value if v in labels)
        self.fanout_select.options = labels
        self.fanout_select.value = keep

    def _update_query_tables(self, change):
        snapshot_id = self.label_to_id.get(change["new"])
        if not snapshot_id:
            return
        try:
            info = self._bq_info(snapshot_id)
            self.query_table_dd.options = sorted(info["tables"].keys())
        except Exception as e:
            with self.query_out:
                print(e)

    def _insert_table_ref(self, _):
        snapshot_id = self.label_to_id.get(self.query_snap_cb.value)
        table = self.query_table_dd.value
        if not (snapshot_id and table):
            return
        qualified = f"`{self._bq_info(snapshot_id)['tables'][table]}`"
        sql = self.sql_box.value
        if INSERT_MARKER in sql:
            self.sql_box.value = sql.replace(INSERT_MARKER, qualified, 1)
        else:
            self.sql_box.value = sql + f" {qualified} "

    def _fanout_jobs(self):
        """Yield (snapshot_label, sql) pairs for the fan-out selection."""
        sql_template = self.sql_box.value
        if FANOUT_PLACEHOLDER not in sql_template:
            raise ValueError(
                f"Fan-out mode requires {FANOUT_PLACEHOLDER} in the SQL.")
        if not self.fanout_select.value:
            raise ValueError(f"Select at least one {self.noun} to fan out "
                             "over.")
        for label in self.fanout_select.value:
            info = self._bq_info(self.label_to_id[label])
            dataset = f"{info['projectId']}.{info['datasetName']}"
            yield label, sql_template.replace(FANOUT_PLACEHOLDER, dataset)

    def _run_query(self, dry: bool):
        fanout = self.query_mode.value.startswith("Fan-out")
        with self.query_out:
            self.query_out.clear_output()
            try:
                jobs = list(self._fanout_jobs()) if fanout \
                    else [("query", self.sql_box.value)]
            except Exception as e:
                print(e)
                return

            # Header: make it unambiguous what was run, from which mode
            mode_desc = (f"Fan-out across {len(jobs)} {self.noun}(s)"
                         if fanout else "Single query")
            sql_shown = self.sql_box.value.strip()
            if len(sql_shown) > 500:
                sql_shown = sql_shown[:500] + " …"
            print(f"=== {'DRY RUN' if dry else 'RUN'} · {mode_desc} ===")
            print(sql_shown)
            print("=" * 40)

            log_kind = ("dry run · fan-out" if fanout else "dry run") \
                if dry else ("query · fan-out" if fanout else "query")
            log_sql = self.sql_box.value.strip()

            if dry:
                total, zero_seen, valid = 0, False, 0
                invalid = []  # (label, error message)
                cfg = self._job_config(dry_run=True, use_query_cache=False)
                for label, sql in jobs:
                    try:
                        job = self.bq.query(sql, job_config=cfg)
                        nbytes = job.total_bytes_processed or 0
                        total += nbytes
                        valid += 1
                        zero_seen = zero_seen or nbytes == 0
                        print(f"  {label}: {nbytes / 1e9:,.3f} GB")
                    except Exception as e:
                        invalid.append((label, str(e)))
                        print(f"  {label}: INVALID — {e}")
                if not valid:
                    print("\nNo valid queries.")
                    self._log_with_failures(log_kind, log_sql, 0, invalid)
                    return
                if invalid:
                    print(f"\nSQL is valid for {valid} of {len(jobs)} "
                          f"{self.noun}(s); the {len(invalid)} listed above "
                          "would be skipped at run time (a real Run "
                          "tolerates per-" + self.noun + " failures the "
                          "same way).")
                    print(f"Total over valid {self.noun}(s): "
                          f"{_estimate_cost(total)}")
                else:
                    print(f"\nSQL is valid. Total {_estimate_cost(total)}")
                if zero_seen:
                    print("\n⚠ A 0.000 GB estimate does NOT mean free: for "
                          "external-table-backed data (as TDR datasets "
                          "are), BigQuery can't estimate scan size before "
                          "execution and reports a 0-byte lower bound. "
                          "The query will still incur normal per-byte "
                          "charges; the actual scan size is shown after "
                          "each real run and recorded in the Log.")
                self._log_with_failures(log_kind, log_sql, valid, invalid,
                                        gb=total / 1e9)
                return

            print(f"Running {len(jobs)} quer"
                  f"{'y' if len(jobs) == 1 else 'ies'} "
                  f"(download capped at {self.max_download_rows:,} rows "
                  "per query) ...")
            frames, capped_info = [], []
            failures = []  # (label, error message)
            scanned_bytes = 0

            def run_one(label, sql):
                job = self.bq.query(sql, job_config=self._job_config())
                result = job.result()          # query completes in BigQuery
                total_rows = result.total_rows  # known BEFORE downloading
                if total_rows > self.max_download_rows:
                    result = job.result(max_results=self.max_download_rows)
                df = result.to_dataframe()
                if fanout:
                    df.insert(0, f"{self.noun}_id",
                              self.label_to_id.get(label, ""))
                    df.insert(0, f"{self.noun}_name",
                              self.label_to_name.get(label, label))
                return label, df, total_rows, job.total_bytes_processed or 0

            with ThreadPoolExecutor(max_workers=4) as pool:
                futures = {pool.submit(run_one, lb, sq): lb
                           for lb, sq in jobs}
                for fut in as_completed(futures):
                    label = futures[fut]
                    try:
                        lb, df, total_rows, nbytes = fut.result()
                        frames.append(df)
                        scanned_bytes += nbytes
                        if total_rows > len(df):
                            capped_info.append((lb, total_rows))
                            print(f"  done: {lb} (downloaded {len(df):,} "
                                  f"of {total_rows:,} rows — CAPPED)")
                        else:
                            print(f"  done: {lb} ({len(df):,} rows)")
                    except Exception as e:
                        failures.append((label, str(e)))
                        print(f"  FAILED: {label} — {e}")

            if not frames:
                print("No results.")
                self._log_with_failures(log_kind, log_sql, 0, failures)
                return
            self.result_df = pd.concat(frames, ignore_index=True, sort=False)
            self._last_capped = bool(capped_info)
            self.export_btn.disabled = False
            self.export_btn.description = \
                f"Export result ({len(self.result_df):,} rows)…"
            if self.export_panel.layout.display != "none":
                self._refresh_export_panel_state()
            extra = (f"; {len(capped_info)} download(s) capped"
                     if capped_info else "")
            self._log_with_failures(log_kind, log_sql + extra, len(frames),
                                    failures, rows=len(self.result_df),
                                    gb=scanned_bytes / 1e9)
            print(f"\nCombined: {len(self.result_df):,} rows × "
                  f"{len(self.result_df.columns)} columns "
                  f"(available as .result_df)"
                  + (f" — {len(failures)} {self.noun}(s) failed"
                     if failures else ""))
            print(f"Actual scan: {_estimate_cost(scanned_bytes)}")
            if capped_info:
                print(f"\n⚠ Results were truncated to protect notebook "
                      f"memory. Full sizes: "
                      + ", ".join(f"{lb}: {n:,}" for lb, n in capped_info)
                      + ".\n  Options: aggregate/filter in SQL so BigQuery "
                        "does the heavy lifting; use 'Run → bucket' "
                        "to export the FULL result server-side (never "
                        "touches notebook memory); or relaunch with "
                        "TdrExplorer(max_download_rows=...) if your VM has "
                        "the RAM.")
            if len(self.result_df) > self.preview_rows:
                print(f"\nPreviewing first {self.preview_rows:,} of "
                      f"{len(self.result_df):,} rows below — the full "
                      "result is available via 'Export result…' or "
                      "app.result_df (preview size is adjustable: "
                      "TdrExplorer(preview_rows=...))")
            with pd.option_context("display.max_rows",
                                   self.preview_rows + 1,
                                   "display.min_rows",
                                   self.preview_rows + 1):
                display(self.result_df.head(self.preview_rows))

    def _run_to_bucket(self, _):
        """Run the query and have BigQuery write the FULL result directly
        to the workspace bucket (EXPORT DATA) — no notebook memory used."""
        fanout = self.query_mode.value.startswith("Fan-out")
        with self.query_out:
            self.query_out.clear_output()
            path = self.bucket_path.value.strip().rstrip("/")
            if not path:
                stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
                path = f"exports/query_{stamp}"
            err = validate_gcs_path(path)
            if err:
                print(f"Invalid destination folder: {err}")
                return
            if path.startswith("gs://"):
                base = path
            elif WORKSPACE_BUCKET:
                base = f"{WORKSPACE_BUCKET.rstrip('/')}/{path.lstrip('/')}"
            else:
                print("No workspace bucket detected — provide a full "
                      "gs:// destination instead.")
                return
            try:
                jobs = list(self._fanout_jobs()) if fanout \
                    else [("query", self.sql_box.value)]
            except Exception as e:
                print(e)
                return

            mode_desc = (f"Fan-out across {len(jobs)} {self.noun}(s)"
                         if fanout else "Single query")
            fmt_map = {"CSV (gzip)": ("CSV", "GZIP", "csv.gz"),
                       "JSON (gzip)": ("JSON", "GZIP", "json.gz"),
                       "Parquet": ("PARQUET", None, "parquet")}
            fmt, compression, ext = fmt_map[self.bucket_fmt_dd.value]
            print(f"=== RUN → BUCKET ({self.bucket_fmt_dd.value}) · "
                  f"{mode_desc} ===")
            print(f"Destination: {base}/\n")

            def _export_sql(uri, f, comp, sql, header=True):
                opts = [f"uri='{uri}'", f"format='{f}'"]
                if comp:
                    opts.append(f"compression='{comp}'")
                if f == "CSV":
                    opts.append(f"header={'true' if header else 'false'}")
                opts.append("overwrite=true")
                return (f"EXPORT DATA OPTIONS({', '.join(opts)}) AS\n"
                        + sql.strip().rstrip(";"))

            def export_one(label, sql):
                safe = "".join(c if c.isalnum() or c in "-_" else "_"
                               for c in label)[:60] or "query"
                want_single = self.bucket_single_cb.value
                f, comp, extension = fmt, compression, ext
                notes = []

                # Check the schema BEFORE exporting: CSV can't hold ARRAY
                # or STRUCT columns, and TDR schemas use them heavily. It
                # is better to say so up front, naming the columns, than
                # to let the export fail and silently change format.
                eff_sql = sql
                if f == "CSV":
                    bad = self._csv_unsafe_fields(sql)
                    if bad:
                        names = [b[0] for b in bad]
                        shown = ", ".join(names[:5])
                        if len(names) > 5:
                            shown += f", +{len(names) - 5} more"
                        if self.bucket_flatten_cb.value:
                            delim = self.bucket_delim.value or "|"
                            eff_sql = self._flatten_sql(sql, bad, delim)
                            has_struct = any(
                                t in ("RECORD", "STRUCT") for _, t, _ in bad)
                            notes.append(
                                f"flattened to keep CSV: {shown} "
                                f"(arrays joined with '{delim}'"
                                + ("; structs as JSON" if has_struct else "")
                                + ")")
                        else:
                            f, comp, extension = "PARQUET", None, "parquet"
                            notes.append(
                                f"CSV can't represent these ARRAY/STRUCT "
                                f"columns: {shown}. Exported as PARQUET "
                                "instead — tick 'Flatten ARRAY/STRUCT "
                                "columns' to keep CSV.")

                for _ in range(2):   # at most one ARRAY→Parquet fallback
                    # Parquet is a footer-based format: shards can't be
                    # concatenated, so a single file isn't possible.
                    merge = want_single and f != "PARQUET"
                    parquet_note = ("single-file merge skipped: Parquet "
                                    "shards can't be concatenated")
                    if want_single and f == "PARQUET" \
                            and parquet_note not in notes:
                        notes.append(parquet_note)
                    uri = f"{base}/{safe}_*.{extension}"
                    try:
                        job = self.bq.query(
                            _export_sql(uri, f, comp, eff_sql,
                                        header=not merge),
                            job_config=self._job_config())
                        job.result()
                        nbytes = job.total_bytes_processed or 0
                    except Exception as e:
                        # Safety net if the schema check missed something
                        if f == "CSV" and _is_array_csv_error(e):
                            f, comp, extension = "PARQUET", None, "parquet"
                            eff_sql = sql   # export the data as-is
                            notes.append("CSV export rejected by BigQuery "
                                         "(ARRAY/STRUCT columns) — "
                                         "exported as PARQUET instead")
                            continue
                        raise
                    if not merge:
                        return label, uri, nbytes, notes, f
                    # The export already succeeded; a merge failure must
                    # not lose it, so fall back to reporting the shards.
                    hdr = ("; the CSV shards have no header row — column "
                           "names are in "
                           f"{base}/{safe}__header.csv.gz"
                           if f == "CSV" else "")
                    try:
                        final, warn = self._combine_shards(
                            base, safe, extension, f, eff_sql)
                        if warn:
                            notes.append(warn)
                        return label, final, nbytes, notes, f
                    except _MergeTooLarge as te:
                        notes.append(f"{te} — left as shards, which stay "
                                     f"readable by wildcard{hdr}")
                        return label, uri, nbytes, notes, f
                    except Exception as ce:
                        notes.append(f"exported, but merging into one "
                                     f"file failed ({str(ce)[:120]}){hdr}")
                        return label, uri, nbytes, notes, f
                raise RuntimeError("export retries exhausted")

            total_bytes, ok = 0, 0
            failures, notes = [], []  # (label, message) each
            fmt_changed = []
            with ThreadPoolExecutor(max_workers=4) as pool:
                futures = {pool.submit(export_one, lb, sq): lb
                           for lb, sq in jobs}
                for fut in as_completed(futures):
                    label = futures[fut]
                    try:
                        lb, uri, nbytes, item_notes, used_fmt = fut.result()
                        total_bytes += nbytes
                        ok += 1
                        if used_fmt != fmt:
                            fmt_changed.append(lb)
                        for msg in item_notes:
                            notes.append((lb, msg))
                        print(f"  done: {lb} → {uri}")
                        for msg in item_notes:
                            print(f"        note: {msg}")
                    except Exception as e:
                        failures.append((label, str(e)))
                        print(f"  FAILED: {label} — {e}")

            if fmt_changed:
                print(f"\n⚠ FORMAT CHANGED: you chose "
                      f"{self.bucket_fmt_dd.value}, but "
                      f"{len(fmt_changed)} export(s) were written as "
                      "PARQUET — see the notes above for which columns "
                      "forced it.")

            detail = self.sql_box.value.strip()
            detail += f"\n-- DESTINATION --\n{base}/"
            if notes:
                detail += "\n-- NOTES --\n" + "\n".join(
                    f"{lb}: {msg}" for lb, msg in notes)
            self._log_with_failures("query → bucket", detail,
                                    ok, failures, gb=total_bytes / 1e9)
            if ok:
                print(f"\n{ok} export(s) written under {base}/. "
                      f"Scanned {_estimate_cost(total_bytes)}.\n"
                      f"List them with: !gsutil ls {base}/")
                if not self.bucket_single_cb.value:
                    print("Tick 'Merge into a single file' above if you'd "
                          "rather have one file per query than many "
                          "shards.")

    # -- Tab 4: export -------------------------------------------------------

    def _build_export_panel(self):
        """The export UI, shown as a collapsible panel on the Query tab
        (opened by the 'Export result…' button after a successful run)."""
        self.export_mode = w.RadioButtons(
            options=["File in workspace bucket", "Terra data table"],
            description="Target:")
        self.export_path = w.Text(
            value="exports/query_result.csv",
            description="File path:",
            tooltip="Path within the workspace bucket, or a full gs:// URL. "
                    "Format inferred from extension: "
                    ".csv, .tsv, .parquet, .json",
            layout=w.Layout(width="60%"))
        self.entity_name = w.Text(
            value="query_result", description="Table name:",
            tooltip="Entity type name for the Terra data table")
        self.dt_namespace = w.Text(
            value=WORKSPACE_NAMESPACE, description="Namespace:",
            tooltip="Target workspace namespace (billing project). "
                    "Pre-filled with the current workspace; edit to "
                    "export to a different workspace you can write to.")
        self.dt_workspace = w.Text(
            value=WORKSPACE_NAME, description="Workspace:",
            tooltip="Target workspace name. Pre-filled with the current "
                    "workspace; edit to export to a different workspace "
                    "you can write to.")
        self.id_col_dd = w.Dropdown(
            description="ID column:", options=["(auto row index)"],
            tooltip="Column to use as the entity ID for data tables")
        do_export_btn = w.Button(description="Export now", icon="save",
                                 button_style="warning")
        close_btn = w.Button(description="Close")
        self.export_out = w.Output()
        self.export_capped_html = w.HTML("")
        self.export_what_html = w.HTML("")

        do_export_btn.on_click(self._export)
        close_btn.on_click(
            lambda _: setattr(self.export_panel.layout, "display", "none"))

        bucket_hint = w.HTML(
            f"<i>Workspace bucket: <code>{WORKSPACE_BUCKET or 'not set'}"
            "</code>. Format is inferred from the extension: .csv, .tsv, "
            ".parquet, or .json (newline-delimited). ARRAY columns need "
            "Parquet or JSON to survive intact. Relative paths are "
            "written under the bucket; "
            "subdirectories are fine. A full <code>gs://</code> URL "
            "writes to any bucket your Terra identity can write to, "
            "including other workspaces' buckets.</i>")
        dt_hint = w.HTML(
            "<i>Pre-filled with the current workspace — edit to send the "
            "table to any Terra workspace where you have writer access.</i>")

        # Group the target-specific fields so only the relevant ones show
        self.csv_fields = w.VBox([bucket_hint, self.export_path])
        self.dt_fields = w.VBox([dt_hint,
                                 w.HBox([self.dt_namespace,
                                         self.dt_workspace]),
                                 w.HBox([self.entity_name,
                                         self.id_col_dd])])
        self.dt_fields.layout.display = "none"

        def toggle_export_fields(change):
            csv = change["new"].startswith("File")
            self.csv_fields.layout.display = "" if csv else "none"
            self.dt_fields.layout.display = "none" if csv else ""
        self.export_mode.observe(toggle_export_fields, names="value")

        self.export_panel = w.VBox(
            [self.export_what_html, self.export_capped_html,
             self.export_mode, self.csv_fields, self.dt_fields,
             w.HBox([do_export_btn, close_btn]), self.export_out],
            layout=w.Layout(border="1px solid #5cb85c", padding="8px",
                            margin="4px 0"))
        self.export_panel.layout.display = "none"
        return self.export_panel

    def _refresh_export_panel_state(self):
        """Keep the export panel's dynamic bits in sync with the latest
        result: ID column choices, description, and capped warning."""
        cols = ["(auto row index)"]
        if self.result_df is not None:
            cols += list(self.result_df.columns)
        self.id_col_dd.options = cols
        n = 0 if self.result_df is None else len(self.result_df)
        self.export_what_html.value = (
            f"<i>Exports the downloaded result of the last run shown "
            f"above ({n:,} rows).</i>")
        self.export_capped_html.value = (
            "<b>⚠ That result was truncated by the download cap — this "
            "export will NOT contain the full result set. Use "
            "'Run → bucket…' for a complete server-side export.</b>"
            if getattr(self, "_last_capped", False) else "")

    def _toggle_export_panel(self, _):
        if self.export_panel.layout.display != "none":
            self.export_panel.layout.display = "none"
            return
        self._refresh_export_panel_state()
        self.export_panel.layout.display = ""

    def _export(self, _):
        with self.export_out:
            self.export_out.clear_output()
            if self.result_df is None or self.result_df.empty:
                print("No query results to export — run a query first.")
                return
            if self.export_mode.value.startswith("File"):
                self._export_file()
            else:
                self._export_data_table(
                    self.entity_name.value.strip() or "query_result")

    def _export_file(self):
        path = self.export_path.value.strip() or "query_result.csv"
        err = validate_gcs_path(path)
        if err:
            print(f"Invalid path: {err}")
            return
        if path.startswith("gs://"):
            dest = path
        elif WORKSPACE_BUCKET:
            dest = f"{WORKSPACE_BUCKET.rstrip('/')}/{path.lstrip('/')}"
        else:
            dest = None
        local = os.path.basename(path) or "query_result.csv"

        # Format is inferred from the file extension
        ext = os.path.splitext(local)[1].lower()
        n = len(self.result_df)
        try:
            if ext == ".tsv":
                self.result_df.to_csv(local, sep="\t", index=False)
            elif ext == ".parquet":
                self.result_df.to_parquet(local, index=False)
            elif ext in (".json", ".jsonl", ".ndjson"):
                self.result_df.to_json(local, orient="records", lines=True)
            elif ext == ".csv":
                self.result_df.to_csv(local, index=False)
            else:
                local += ".csv"
                dest = dest + ".csv" if dest else None
                self.result_df.to_csv(local, index=False)
                print(f"No recognized extension "
                      "(.csv/.tsv/.parquet/.json) — wrote CSV as "
                      f"{local}")
        except Exception as e:
            print(f"Failed to write {ext or 'file'}: {e}")
            if ext == ".parquet":
                print("Parquet needs pyarrow (usually preinstalled in "
                      "Terra). Note that ARRAY columns export cleanly to "
                      "Parquet but are stringified in CSV/TSV.")
            self._log(f"export · {ext.lstrip('.') or 'file'}", path,
                      f"failed: {e}"[:200], rows=n)
            return

        kind = f"export · {os.path.splitext(local)[1].lstrip('.')}"
        if dest is None:
            print(f"No workspace bucket detected; wrote ./{local}")
            self._log(kind, f"./{local}", "local only", rows=n)
            return
        # -u bills the request to the workspace project, which makes
        # copies to requester-pays buckets in other projects work too
        u_flag = f"-u '{GOOGLE_PROJECT}' " if GOOGLE_PROJECT else ""
        rc = os.system(f"gsutil {u_flag}cp '{local}' '{dest}'")
        if rc == 0:
            print(f"Wrote {n:,} rows to {dest}")
            self._log(kind, dest, "ok", rows=n)
        else:
            print(f"gsutil copy failed (exit {rc}); "
                  f"file kept locally at ./{local}")
            self._log(kind, dest, f"failed (gsutil exit {rc})", rows=n)

    def _export_data_table(self, entity_type: str):
        if len(self.result_df) > 50_000:
            print(f"Result has {len(self.result_df):,} rows — Terra data "
                  "tables aren't designed for that scale, and the upload "
                  "will likely time out or make the workspace Data tab "
                  "unusable. Export to a CSV in the bucket instead, or "
                  "aggregate the result down first.")
            return
        df = self.result_df.copy()
        id_col = self.id_col_dd.value
        id_header = f"entity:{entity_type}_id"
        if id_col != "(auto row index)" and id_col in df.columns:
            df.insert(0, id_header, df.pop(id_col).astype(str))
        else:
            width = len(str(len(df)))
            df.insert(0, id_header,
                      [f"{i:0{width}d}" for i in range(1, len(df) + 1)])
        if df[id_header].duplicated().any():
            print("Entity IDs must be unique — the chosen ID column "
                  "has duplicates.")
            return

        tsv = df.to_csv(sep="\t", index=False)
        target_ns = self.dt_namespace.value.strip() or WORKSPACE_NAMESPACE
        target_ws = self.dt_workspace.value.strip() or WORKSPACE_NAME
        if not (target_ns and target_ws):
            print("Target workspace namespace and name are required.")
            return
        dt_detail = (f"data table '{entity_type}' in "
                     f"{target_ns}/{target_ws}")
        try:
            import firecloud.api as fapi
            resp = fapi.upload_entities(target_ns, target_ws,
                                        tsv, model="flexible")
            if resp.status_code in (200, 201):
                print(f"Created/updated {len(df):,} rows in data table "
                      f"'{entity_type}' in {target_ns}/{target_ws}")
                self._log("export · data table", dt_detail, "ok",
                          rows=len(df))
            else:
                print(f"Upload failed ({resp.status_code}): {resp.text[:500]}")
                if resp.status_code in (403, 404):
                    print("Check that the target workspace exists and that "
                          "you have writer access to it.")
                self._log("export · data table", dt_detail,
                          f"failed ({resp.status_code})", rows=len(df))
        except ImportError:
            # Fall back to the Orchestration API directly
            resp = requests.post(
                "https://api.firecloud.org/api/workspaces/"
                f"{target_ns}/{target_ws}/flexibleImportEntities",
                headers={"Authorization": f"Bearer {_access_token()}"},
                files={"entities": ("entities.tsv", tsv,
                                    "text/tab-separated-values")},
            )
            if resp.ok:
                print(f"Created/updated {len(df):,} rows in data table "
                      f"'{entity_type}' in {target_ns}/{target_ws}")
                self._log("export · data table", dt_detail, "ok",
                          rows=len(df))
            else:
                print(f"Upload failed ({resp.status_code}): {resp.text[:500]}")
                if resp.status_code in (403, 404):
                    print("Check that the target workspace exists and that "
                          "you have writer access to it.")
                self._log("export · data table", dt_detail,
                          f"failed ({resp.status_code})", rows=len(df))

    # -- Tab 5: log ------------------------------------------------------------

    def _build_log_tab(self):
        refresh_btn = w.Button(description="Refresh", icon="refresh",
                               button_style="primary")
        save_btn = w.Button(description="Save log to bucket", icon="save",
                            tooltip="Write the full log as CSV to "
                                    "exports/tdr_explorer_log.csv in the "
                                    "workspace bucket")
        clear_btn = w.Button(description="Clear log", icon="trash",
                             button_style="danger",
                             tooltip="Clears the in-memory log AND the "
                                     "persisted log file")
        hint = w.HTML(
            f"<i>Every query, dry run, and export is recorded here and "
            f"persisted to <code>{LOG_PATH}</code> on the environment's "
            "disk, so the log survives pause/resume. Also available as "
            "<code>app.log_df</code>.</i>")
        self.log_out = w.Output(
            layout=w.Layout(max_height="500px", overflow="auto",
                            border="1px solid #ddd"))

        def show_log(_=None):
            with self.log_out:
                self.log_out.clear_output()
                df = self.log_df
                if df.empty:
                    print("Nothing logged yet.")
                    return
                print(f"{len(df)} entr{'y' if len(df) == 1 else 'ies'} "
                      "(newest first)")
                with pd.option_context("display.max_rows", 100,
                                       "display.max_colwidth", 120):
                    display(df)

        def save_log(_):
            with self.log_out:
                self.log_out.clear_output()
                if not self.log:
                    print("Nothing to save.")
                    return
                local = "tdr_explorer_log.csv"
                self.log_df.to_csv(local, index=False)
                if not WORKSPACE_BUCKET:
                    print(f"No workspace bucket; wrote ./{local}")
                    return
                dest = (f"{WORKSPACE_BUCKET.rstrip('/')}/exports/"
                        f"tdr_explorer_log.csv")
                rc = os.system(f"gsutil cp '{local}' '{dest}'")
                print(f"Wrote log to {dest}" if rc == 0 else
                      f"gsutil copy failed (exit {rc}); wrote ./{local}")

        def clear_log(_):
            self.log = []
            try:
                os.remove(LOG_PATH)
            except FileNotFoundError:
                pass
            show_log()

        refresh_btn.on_click(show_log)
        save_btn.on_click(save_log)
        clear_btn.on_click(clear_log)
        self._show_log = show_log  # so show() can render initial state
        return w.VBox([hint, w.HBox([refresh_btn, save_btn, clear_btn]),
                       self.log_out])

    # -- assembly ------------------------------------------------------------

    def _build_ui(self):
        tabs = w.Tab(children=[
            self._build_snapshots_tab(),
            self._build_schema_tab(),
            self._build_query_tab(),
            self._build_log_tab(),
        ])
        for i, title in enumerate([self.Noun + "s", "Schema", "Query",
                                   "Log"]):
            tabs.set_title(i, title)
        header = w.HTML(
            f"<h3>TDR {self.Noun} Explorer</h3>"
            f"<p>Billing project: <code>{self.billing_project or 'not set'}</code> · "
            f"Workspace: <code>{WORKSPACE_NAMESPACE}/{WORKSPACE_NAME}</code></p>")
        self.ui = w.VBox([header, tabs])

    def show(self):
        # In classic Jupyter, tall cell outputs get clipped into a small
        # scroll box. Disable that for this notebook so the full app shows;
        # the app's own output panes are individually height-bounded instead.
        # (No-op in JupyterLab, which doesn't auto-clip.)
        try:
            from IPython.display import Javascript
            display(Javascript(
                "if (typeof IPython !== 'undefined' && IPython.OutputArea)"
                "{IPython.OutputArea.prototype._should_scroll = "
                "function(){return false};}"))
        except Exception:
            pass
        display(self.ui)
        with self.snap_out:
            print("Enter an optional keyword and click Search to list your "
                  f"{self.mode} (blank filter = all {self.mode}).")
        self._show_log()  # render any log entries persisted from past sessions
        return self


## Launch the app

In [ ]:
app = TdrExplorer().show()